In [1]:
import torch
import ase
import torch_sim as ts
from ase import Atoms
import numpy as np
from torch_sim import fire_init, fire_step, gradient_descent_init, gradient_descent_step
from fumi_tosi import FumiTosiModel
from torch_sim.models.lennard_jones import LennardJonesModel
# Based on combination of these:
# https://pubs.aip.org/aip/jcp/article/130/4/044505/932975/Thermal-conductivity-of-molten-alkali-halides
# https://www.sciencedirect.com/science/article/pii/S2352152X22007204

# LiCl Salt Setup
a = torch.tensor([0.421999990940094, 0.29010000824928284, 0.1581999957561493], device=torch.device('cuda'))
c = torch.tensor([0.045570001006126404, 1.2484999895095825, 69.29000091552734], device=torch.device('cuda'))
d = torch.tensor([0.018727000802755356, 1.4982000589370728, 139.2100067138672], device=torch.device('cuda'))
licl_model = FumiTosiModel(
    atomic_number_zi=3,                                # Atomic number of Li
    ionic_charge_i= 1,                                 # ionic charge of Li
    ionic_charge_j= -1,                                # ionic charge of Cl
    sigma_ij= torch.tensor([1.632, 2.401, 3.170]),     # sigma_ij (in Å) for Li-Li, Li-Cl & Cl-Cl pairs
    a=a,                                               # Aij (in eV)
    b= torch.tensor([2.9200], device=torch.device('cuda')),                         # B (in Å^(-1))
    c=c,                                                # Cij (in eV * Å^(6))
    d=d,                                                # Dij (in eV * Å^8)
    rc=7,                                               # radial cutoff (in Å)
    device=torch.device('cuda'),
    dtype=torch.float64,
    compute_forces=True,
    compute_stress=True,
    per_atom_energies=True,
    per_atom_stresses=True,
    use_neighbor_list=True,
)

# Create a system with 20 LiCl particles
positions = torch.randint(0, 10, (20,3)) * torch.rand(20, 3)
symbols = ['Li']*10 + ['Cl']*10
cell = [10,10,10]
atoms = Atoms(symbols=symbols, positions=positions, cell=cell, pbc=True)
licl_system = ts.initialize_state(atoms, device=torch.device('cuda'), dtype=torch.float64)

licl_system = ts.integrate(
    system=licl_system,
    model= licl_model,
    n_steps=5000,
    temperature=600, # in kelvin
    timestep=0.00001, # in ps
    integrator=ts.Integrator.nve,
)
output = licl_model(licl_system)
# Output has the following keys:
# 'energy': total potential energy of the system, 'energies': atom-wise potential energy
# 'forcees': per-atom force, 'stress': stress on the system, 'stresses': per-atom stress.
print(output)

licl_system = ts.runners.optimize(
    system=licl_system,
    model = licl_model,
    optimizer= (fire_init, fire_step),
    convergence_fn=ts.runners.generate_energy_convergence_fn(energy_tol=1e-6),
)
preoptimization_energy = output['energy'].item()
postoptimization_energy = licl_model(licl_system)['energy'].item()
print("Did we optimize the system?", postoptimization_energy < preoptimization_energy)
print("This is preoptimization energy", preoptimization_energy)
print("This is postoptimization energy", postoptimization_energy)

Warp DeprecationWarning: The symbol `warp.vec` will soon be removed from the public API. Use `warp.types.vector` instead.
{'energy': tensor([28.5167], device='cuda:0', dtype=torch.float64), 'energies': tensor([0.7800, 0.2270, 4.5037, 1.5721, 8.2157, 0.0285, 0.7540, 0.2687, 6.1453,
        0.0950, 0.3344, 7.7302, 0.5061, 7.8993, 1.7272, 1.3239, 0.9129, 4.8024,
        4.4558, 4.7510], device='cuda:0', dtype=torch.float64), 'forces': tensor([[-2.9048e+00,  1.8486e+00,  1.6319e+00],
        [-8.3624e-01,  1.9346e-01,  9.7907e-01],
        [-2.4723e+01,  5.3607e+00,  5.5213e-01],
        [ 5.6981e+00,  2.8584e+00,  9.0360e-01],
        [ 8.6840e+00,  3.6120e+01, -4.4847e-01],
        [-3.6133e-03, -9.8916e-02,  9.5993e-02],
        [ 1.1818e+00, -3.3378e+00,  1.3215e+00],
        [ 9.2618e-01,  2.9122e-02, -8.4377e-01],
        [ 9.8839e-02, -1.7228e+01, -2.5995e+01],
        [ 3.7741e-01, -2.6092e-01, -1.4613e-01],
        [ 2.0696e+00,  5.5504e-01, -8.3409e-01],
        [-1.0765e+00,  2.